In [9]:
! pip install ftfy

In [10]:
import numpy as np
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/ashery/chexpert/train.csv")

train_df["Path"] = train_df["Path"].str.replace(
    "CheXpert-v1.0-small/train/",
    "/kaggle/input/datasets/ashery/chexpert/train/",
    regex=False
)

label_cols = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

train_df = train_df[train_df["Frontal/Lateral"] == "Frontal"].copy()

train_df[label_cols] = train_df[label_cols].fillna(0).astype(int)
train_df = train_df.reset_index(drop=True)
train_df["img_id"] = train_df.index

print(train_df[["img_id", "Path"]].head(2))

#eval_df = train_df.reset_index(drop=True)
SEED = 42
N_SAMPLE = 10000
#print(f"Total frontal images: {len(eval_df):,}")
print("\nPositive cases per finding:")

rng = np.random.default_rng(SEED)
sampled_indices = rng.choice(train_df.index, size=N_SAMPLE, replace=False)

eval_df = train_df.loc[sorted(sampled_indices)].reset_index(drop=True)

print(f"Sampled evaluation set size: {len(eval_df)}  (seed={SEED})")
for lbl in label_cols:
    print(f"{lbl:<28} {(eval_df[lbl] == 1).sum():,}")
# ---------------- exclude uncertain (-1) labels ----------------
eval_df_masked = eval_df.copy()
for col in label_cols:
    eval_df_masked[col] = eval_df_masked[col].replace(-1, pd.NA)

# ---------------- build GT lookups keyed by img_id (not position) ----------------
gt_positive_by_id = {}
gt_usable_by_id = {}

for _, r in eval_df_masked.iterrows():
    img_id = r["img_id"]
    gt_positive_by_id[img_id] = {
        cond for cond in label_cols if pd.notna(r[cond]) and r[cond] == 1
    }
    gt_usable_by_id[img_id] = {
        cond for cond in label_cols if pd.notna(r[cond])
    }

print("gt_positive_by_id:", len(gt_positive_by_id))
print("gt_usable_by_id:", len(gt_usable_by_id))

# sanity check on a couple of images
sample_id = eval_df["img_id"].iloc[0]
print(f"\nimg_id {sample_id} — positive labels: {sorted(gt_positive_by_id[sample_id])}")
print(f"img_id {sample_id} — usable conditions: {len(gt_usable_by_id[sample_id])}/14")

   img_id                                               Path
0       0  /kaggle/input/datasets/ashery/chexpert/train/p...
1       1  /kaggle/input/datasets/ashery/chexpert/train/p...

Positive cases per finding:
Sampled evaluation set size: 10000  (seed=42)
No Finding                   895
Enlarged Cardiomediastinum   512
Cardiomegaly                 1,214
Lung Opacity                 4,934
Lung Lesion                  400
Edema                        2,588
Consolidation                707
Pneumonia                    248
Atelectasis                  1,520
Pneumothorax                 901
Pleural Effusion             4,083
Pleural Other                114
Fracture                     390
Support Devices              5,583
gt_positive_by_id: 10000
gt_usable_by_id: 10000

img_id 0 — positive labels: ['No Finding', 'Support Devices']
img_id 0 — usable conditions: 14/14


In [ ]:
prevalence = {
    "Pleural Other": 2505, "Pneumonia": 4675, "Lung Lesion": 7040,
    "Fracture": 7436, "Enlarged Cardiomediastinum": 9187, "Consolidation": 12983,
    "No Finding": 16974, "Pneumothorax": 17693, "Cardiomegaly": 23385,
    "Atelectasis": 29720, "Edema": 49675, "Pleural Effusion": 76899,
    "Lung Opacity": 94211, "Support Devices": 107170
}
TOTAL_DATASET = 191027  # or your actual total frontal count
N = 15000

print(f"{'Label':<28}{'Expected count at N=15000':>25}")
for lbl, count in sorted(prevalence.items(), key=lambda x: x[1]):
    expected = count / TOTAL_DATASET * N
    print(f"{lbl:<28}{expected:>25.1f}")

# **Build Candidate pool sentences**

In [ ]:
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.models as tvm
from PIL import Image
from transformers import BertTokenizer, BertModel, BertConfig
import pandas as pd
import spacy
from tqdm import tqdm

# ---------------- build candidate text pool independently ----------------
nlp = spacy.load("en_core_web_sm")

def extract_subsentences(text):
    if not text or text.strip() == "":
        return []
    doc = nlp(text.strip())
    findings = []
    for sent in doc.sents:
        s = sent.text.strip()
        s = " ".join(s.split())
        if len(s.split()) >= 3:
            findings.append(s)
    return findings

mimic_for_pool = pd.read_csv("/kaggle/input/datasets/shery432/mimic-file/mimic-data.csv")
mimic_for_pool['findings'] = mimic_for_pool['findings'].fillna('')
mimic_for_pool = mimic_for_pool[mimic_for_pool['findings'] != ''].reset_index(drop=True)

candidate_pool = set()
for text in tqdm(mimic_for_pool['findings'], desc='Extracting sentences'):
    for sent in extract_subsentences(text):
        candidate_pool.add(sent)

candidate_pool = sorted(candidate_pool)
print(f"Candidate pool size: {len(candidate_pool)}")

Extracting sentences:  14%|█▍        | 4278/30433 [00:58<05:32, 78.76it/s]

In [11]:
len(candidate_pool)

129906

# **My MODEL SETUP**

In [ ]:
import os, sys, subprocess, importlib.util
from collections import OrderedDict
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.models as tvm
from PIL import Image
from transformers import BertTokenizer, BertModel, BertConfig
from tqdm import tqdm

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HIDDEN_DIM = 512
MAX_DIM = 256

label_cols = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

# ---------------- load CheXpert + add img_id ----------------
train_df = pd.read_csv("/kaggle/input/datasets/ashery/chexpert/train.csv")
train_df["Path"] = train_df["Path"].str.replace(
    "CheXpert-v1.0-small/train/",
    "/kaggle/input/datasets/ashery/chexpert/train/",
    regex=False
)
train_df[label_cols] = train_df[label_cols].fillna(0).astype(int)
train_df = train_df[train_df["Frontal/Lateral"] == "Frontal"].copy()
train_df = train_df.reset_index(drop=True)
train_df["img_id"] = train_df.index

# ---------------- GT sets keyed by img_id, uncertain excluded ----------------
eval_df_masked = eval_df.copy()
for col in label_cols:
    eval_df_masked[col] = eval_df_masked[col].replace(-1, pd.NA)

gt_positive_by_id = {}
gt_usable_by_id = {}
for _, r in eval_df_masked.iterrows():
    img_id = r["img_id"]
    gt_positive_by_id[img_id] = {c for c in label_cols if pd.notna(r[c]) and r[c] == 1}
    gt_usable_by_id[img_id]   = {c for c in label_cols if pd.notna(r[c])}

# ---------------- retrieval model ----------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

val_transform = tv.transforms.Compose([
    tv.transforms.Grayscale(num_output_channels=1),
    tv.transforms.Resize((MAX_DIM, MAX_DIM)),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize(mean=[0.5], std=[0.5])
])

def conv1x1(i, o, s=1):
    return nn.Conv2d(i, o, kernel_size=1, stride=s, bias=False)

class SoftAttention(nn.Module):
    def __init__(self, in_groups, m_heads, in_channels):
        super().__init__()
        self.learnable_scalar = nn.Parameter(torch.rand(1))
        self.conv3d = nn.Conv3d(in_groups, m_heads, kernel_size=(in_channels,1,1), stride=(in_channels,1,1))
        self.lrelu = nn.LeakyReLU(inplace=True)
        self.softmax = nn.Softmax(-1)

    def forward(self, x):
        h, w = x.shape[-2], x.shape[-1]
        c = torch.unsqueeze(x, 1)
        c = self.conv3d(c)
        c = self.lrelu(c)
        c = c.squeeze(2)
        c = c.view(c.shape[0], c.shape[1], h*w)
        c = self.softmax(c)
        c = c.view(c.shape[0], c.shape[1], h, w)
        attn_maps = torch.unsqueeze(c.sum(1), 1)
        importance = x * attn_maps
        out = x + importance * self.learnable_scalar.expand_as(importance)
        return out, attn_maps, self.learnable_scalar

class ImageEncoder(nn.Module):
    def __init__(self, output_channels=512):
        super().__init__()
        resnet = tvm.resnet50(weights=None)
        new_conv = nn.Conv2d(1, resnet.conv1.out_channels, kernel_size=resnet.conv1.kernel_size,
                              stride=resnet.conv1.stride, padding=resnet.conv1.padding, bias=False)
        resnet.conv1 = new_conv
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        self.sa_l3 = SoftAttention(1, 16, 1024)
        self.region = conv1x1(1024, output_channels)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.3)
        self.global_feats = nn.Linear(2048, output_channels)

    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x, attn_l3, scalar_l3 = self.sa_l3(x)
        region_feat = self.region(x)
        x = self.layer4(x); x = self.avgpool(x)
        z = self.dropout(torch.flatten(x, 1))
        global_feat = self.global_feats(z)
        return region_feat, global_feat, attn_l3, scalar_l3, attn_l3, scalar_l3

class MLMHead(nn.Module):
    def __init__(self, bert_config):
        super().__init__()
        self.transform = nn.Linear(bert_config.hidden_size, bert_config.hidden_size)
        self.act = nn.GELU()
        self.LayerNorm = nn.LayerNorm(bert_config.hidden_size, eps=bert_config.layer_norm_eps)
        self.decoder = nn.Linear(bert_config.hidden_size, bert_config.vocab_size - 1)

    def forward(self, seq_feat):
        return self.decoder(self.LayerNorm(self.act(self.transform(seq_feat))))

class TextEncoder(nn.Module):
    def __init__(self, bert_config, output_channels, pool='cls'):
        super().__init__()
        self.pool = pool
        self.bert = BertModel(bert_config, add_pooling_layer=False)
        self.mlm = MLMHead(bert_config)
        self.sent_fc = nn.Linear(output_channels, output_channels)
        self.word_fc = nn.Linear(output_channels, output_channels)

    def forward(self, x, mask, task='itm'):
        seq_feat = self.bert(input_ids=x, attention_mask=mask)[0]
        if self.pool == 'cls':
            sent_feat = self.sent_fc(seq_feat[:, 0])
        else:
            sent_feat = (seq_feat * mask.unsqueeze(-1)).sum(1) / mask.sum(1).unsqueeze(-1)
            sent_feat = self.sent_fc(sent_feat)
        word_feat = self.word_fc(seq_feat)
        return word_feat.transpose(1, 2), sent_feat

bert_config = BertConfig(
    vocab_size=tokenizer.vocab_size, hidden_size=HIDDEN_DIM, num_hidden_layers=3,
    num_attention_heads=8, intermediate_size=2048, hidden_act='gelu',
    hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
    max_position_embeddings=512, layer_norm_eps=1e-12, initializer_range=0.02,
    type_vocab_size=2, pad_token_id=0
)

IMAGE_ENCODER_PATH = '/kaggle/input/notebooks/shery432/mimic-rag-evaluation/output/MIMIC_sentence_level_2026_07_04_04_48_38/Model/image_encoder_best.pth'
TEXT_ENCODER_PATH  = '/kaggle/input/notebooks/shery432/mimic-rag-evaluation/output/MIMIC_sentence_level_2026_07_04_04_48_38/Model/text_encoder_best.pth'

image_encoder = ImageEncoder(output_channels=HIDDEN_DIM).to(DEVICE)
text_encoder = TextEncoder(bert_config, output_channels=HIDDEN_DIM).to(DEVICE)

img_state = torch.load(IMAGE_ENCODER_PATH, map_location=DEVICE)
image_encoder.load_state_dict(img_state['model'])
txt_state = torch.load(TEXT_ENCODER_PATH, map_location=DEVICE)
text_encoder.load_state_dict(txt_state['model'])
image_encoder.eval(); text_encoder.eval()
print("Loaded checkpoints, epoch:", img_state.get('epoch'), txt_state.get('epoch'))

# ---------------- encode candidate pool ----------------
BATCH = 256
all_sent_feats = []
with torch.no_grad():
    for i in tqdm(range(0, len(candidate_pool), BATCH), desc='Encoding candidate pool'):
        batch_sents = candidate_pool[i:i+BATCH]
        enc = tokenizer(batch_sents, padding='max_length', truncation=True, max_length=64, return_tensors='pt').to(DEVICE)
        _, sent_feats = text_encoder(enc['input_ids'], enc['attention_mask'], task='itm')
        all_sent_feats.append(sent_feats.cpu())
all_sent_feats = torch.cat(all_sent_feats, dim=0)
txt_norm_pool = torch.nn.functional.normalize(all_sent_feats, dim=1).to(DEVICE)

def retrieve_topk(img_path, k=10):
    image = Image.open(img_path).convert('L')
    img_tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, global_feat, _, _, _, _ = image_encoder(img_tensor)
    img_norm = torch.nn.functional.normalize(global_feat, dim=1)
    sims = (img_norm @ txt_norm_pool.t()).squeeze(0)
    topk = torch.topk(sims, k=k)
    sents = [candidate_pool[idx] for idx in topk.indices.tolist()]
    scores = topk.values.tolist()
    return sents, scores

# **CXR-REPAIR SETUP**

In [ ]:
# ── Setup (from your snippet) ──────────────────────────
!git clone https://github.com/rajpurkarlab/CXR-RePaiR.git /kaggle/working/CXR-RePaiR
import sys
sys.path.insert(0, '/kaggle/working/CXR-RePaiR')
import torch
from clip.model import build_model
from clip import clip
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_path = '/kaggle/input/datasets/shery432/cxr-repair/clip-imp-pretrained_128_6_after_4.pt'
repair_state_dict = torch.load(ckpt_path, map_location='cpu')
repair_model = build_model(repair_state_dict).float().to(device).eval()

repair_preprocess = Compose([
    Resize(224, interpolation=Image.BICUBIC),
    CenterCrop(224),
    lambda img: img.convert("RGB"),
    ToTensor(),
    Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
])

def encode_texts_repair(texts, model, device, batch_size=256):
    embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Encoding candidate pool (RePaiR)'):
            batch = texts[i:i+batch_size]
            toks = clip.tokenize(batch, context_length=model.context_length).to(device)
            emb = model.encode_text(toks)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embs.append(emb.cpu())
    return torch.cat(embs, dim=0)

def encode_images_repair(image_paths, model, device, batch_size=32):
    embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc='Encoding images (RePaiR)'):
            batch_paths = image_paths[i:i+batch_size]
            imgs = torch.stack([repair_preprocess(Image.open(p).convert('L')) for p in batch_paths]).to(device)
            emb = model.encode_image(imgs)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embs.append(emb.cpu())
    return torch.cat(embs, dim=0)

# ── Encode candidate pool (text) ────────────────────────
txt_norm_pool_repair = encode_texts_repair(candidate_pool, repair_model, device).to(device)

# ── Single-image top-k retrieval ────────────────────────
def retrieve_topk_repair(img_path, k=10):
    img_emb = encode_images_repair([img_path], repair_model, device, batch_size=1).to(device)
    sims = (img_emb @ txt_norm_pool_repair.t()).squeeze(0)
    topk = torch.topk(sims, k=k)
    sents = [candidate_pool[idx] for idx in topk.indices.tolist()]
    scores = topk.values.tolist()
    return sents, scores

# **CXR-REDONE SETUP**

In [14]:
import torch
import torch.nn as nn
import transformers.modeling_utils as _modeling_utils

# apply_chunking_to_forward (you already added this — kept here for completeness)
if not hasattr(_modeling_utils, "apply_chunking_to_forward"):
    import inspect
    def apply_chunking_to_forward(forward_fn, chunk_size, chunk_dim, *input_tensors):
        tensor_shape = input_tensors[0].shape[chunk_dim]
        num_args = len(inspect.signature(forward_fn).parameters)
        if chunk_size > 0:
            num_chunks = input_tensors[0].shape[chunk_dim] // chunk_size
            chunks = tuple(t.chunk(num_chunks, dim=chunk_dim) for t in input_tensors)
            outputs = tuple(forward_fn(*chunk_set) for chunk_set in zip(*chunks))
            return torch.cat(outputs, dim=chunk_dim)
        return forward_fn(*input_tensors)
    _modeling_utils.apply_chunking_to_forward = apply_chunking_to_forward

# find_pruneable_heads_and_indices
if not hasattr(_modeling_utils, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _modeling_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

# prune_linear_layer
if not hasattr(_modeling_utils, "prune_linear_layer"):
    def prune_linear_layer(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        b = None
        if layer.bias is not None:
            b = (layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach())
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _modeling_utils.prune_linear_layer = prune_linear_layer

print("Shims installed:",
      hasattr(_modeling_utils, "apply_chunking_to_forward"),
      hasattr(_modeling_utils, "find_pruneable_heads_and_indices"),
      hasattr(_modeling_utils, "prune_linear_layer"))

Shims installed: True True True


In [ ]:
# ---- 4. Download the CXR-ReDonE checkpoint (prior-refs removed) ----------
CXR_REDONE_CKPT = '/kaggle/working/checkpoint_59.pth'
if not os.path.exists(CXR_REDONE_CKPT):
    subprocess.run([
        'wget', '-O', CXR_REDONE_CKPT,
        'https://www.dropbox.com/s/b4tkf2z4v6wa4zj/checkpoint_59.pth?dl=1'
    ], check=True)

In [16]:
# ============================================================
# CXR-ReDonE (ALBEF retrieval, prior-refs-removed) — from scratch
# ============================================================
import os, sys, types, subprocess
import torch
import torch.nn.functional as F
import numpy as np
import yaml
from PIL import Image
from torchvision import transforms
from transformers import BertTokenizer
import transformers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- 1. Clone repo (only need the ALBEF folder) --------------------------
REDONE_DIR = '/kaggle/working/CXR-ReDonE'
if not os.path.exists(REDONE_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/rajpurkarlab/CXR-ReDonE.git', REDONE_DIR], check=True)
ALBEF_DIR = os.path.join(REDONE_DIR, 'ALBEF')

subprocess.run(['pip', 'install', '-q', '--upgrade', 'timm', 'pyyaml', 'h5py'], check=True)

try:
    import transformers.file_utils as _futils
except ModuleNotFoundError:
    _futils = types.ModuleType('transformers.file_utils')
    sys.modules['transformers.file_utils'] = _futils
    transformers.file_utils = _futils

def _noop_decorator_factory(*a, **k):
    def _wrap(fn):
        return fn
    return _wrap

for _name in ['add_code_sample_docstrings', 'add_start_docstrings',
              'add_start_docstrings_to_model_forward', 'replace_return_docstrings']:
    setattr(_futils, _name, _noop_decorator_factory)
if not hasattr(_futils, 'ModelOutput'):
    from transformers.utils import ModelOutput as _ModelOutput
    _futils.ModelOutput = _ModelOutput

# vit.py's register_model decorator
try:
    import timm.models.registry as _tregistry
    if not hasattr(_tregistry, 'register_model'):
        _tregistry.register_model = lambda fn: fn
except ModuleNotFoundError:
    _tregistry = types.ModuleType('timm.models.registry')
    _tregistry.register_model = lambda fn: fn
    sys.modules['timm.models.registry'] = _tregistry

# vit.py's _cfg helper (still present in most timm versions, but just in case)
import timm.models.vision_transformer as _tvit
if not hasattr(_tvit, '_cfg'):
    def _cfg(url='', **kwargs):
        return {'url': url, 'num_classes': 1000, 'input_size': (3, 224, 224), **kwargs}
    _tvit._cfg = _cfg

# ---- 3. Load ALBEF model code (skip the vendored tokenizer entirely) -----
if ALBEF_DIR not in sys.path:
    sys.path.insert(0, ALBEF_DIR)
for m in list(sys.modules):          # drop any stale 'models' module from earlier cells
    if m == 'models' or m.startswith('models.'):
        del sys.modules[m]

from models.model_retrieval import ALBEF as ALBEF_Retrieval
from models.vit import interpolate_pos_embed
from models import xbert as _xbert   # so we can fast-init text_encoder below

_xbert.BertPreTrainedModel.tie_weights = lambda self, *a, **k: None

def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    # newer transformers removed this; xbert never actually uses head_mask here (always None)
    return [None] * num_hidden_layers

_xbert.BertPreTrainedModel.get_head_mask = _get_head_mask
print("ALBEF retrieval code loaded OK")
print("ok")
# ---- 4. Download the CXR-ReDonE checkpoint (prior-refs removed) ----------
CXR_REDONE_CKPT = '/kaggle/working/checkpoint_59.pth'
if not os.path.exists(CXR_REDONE_CKPT):
    subprocess.run([
        'wget', '-O', CXR_REDONE_CKPT,
        'https://www.dropbox.com/s/b4tkf2z4v6wa4zj/checkpoint_59.pth?dl=1'
    ], check=True)


Cloning into '/kaggle/working/CXR-ReDonE'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 61.6 MB/s eta 0:00:00
ALBEF retrieval code loaded OK
ok


/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [17]:
# ---- 5. Build config + model ----------------------------------------------
config = yaml.load(open(os.path.join(ALBEF_DIR, 'configs', 'Retrieval_flickr.yaml')), Loader=yaml.Loader)
config['bert_config'] = os.path.join(ALBEF_DIR, 'configs', 'config_bert.json')
config['image_res'] = 256   # matches the CXR-ReDonE cosine-sim retrieval resolution

redone_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# fast-init: skip downloading pretrained bert-base-uncased weights since the
# checkpoint below overwrites the whole state_dict anyway (same trick as CheXbert)
def _fast_bert_from_pretrained(name, *args, **kwargs):
    cfg = kwargs.get('config')
    add_pooling = kwargs.get('add_pooling_layer', True)
    return _xbert.BertModel(config=cfg, add_pooling_layer=add_pooling)

_orig_xbert_from_pretrained = _xbert.BertModel.from_pretrained
_xbert.BertModel.from_pretrained = _fast_bert_from_pretrained
redone_model = ALBEF_Retrieval(config=config, text_encoder='bert-base-uncased', tokenizer=redone_tokenizer)
_xbert.BertModel.from_pretrained = _orig_xbert_from_pretrained

ckpt = torch.load(CXR_REDONE_CKPT, map_location='cpu')
state_dict = ckpt['model']
state_dict['visual_encoder.pos_embed'] = interpolate_pos_embed(state_dict['visual_encoder.pos_embed'], redone_model.visual_encoder)
state_dict['visual_encoder_m.pos_embed'] = interpolate_pos_embed(state_dict['visual_encoder_m.pos_embed'], redone_model.visual_encoder_m)
for key in list(state_dict.keys()):
    if 'bert' in key:
        state_dict[key.replace('bert.', '')] = state_dict.pop(key)

msg = redone_model.load_state_dict(state_dict, strict=False)
print(msg)
redone_model = redone_model.to(device).eval()

# ---- 6. Image preprocessing (matches CXR_ReDonE_module exactly) -----------
# NOTE: raw pixel values 0-255, no ToTensor()/255 scaling — this matches the
# original h5-based pipeline, just adapted to read from a JPG path like your
# own retrieve_topk_sentences does.
redone_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.BICUBIC),
    transforms.Normalize((101.48761, 101.48761, 101.48761), (83.43944, 83.43944, 83.43944)),
])

def load_image_for_redone(image_path):
    img = Image.open(image_path).convert('L')
    img = np.array(img, dtype=np.float64)
    img = np.expand_dims(img, axis=0)
    img = np.repeat(img, 3, axis=0)
    img = torch.from_numpy(img).float()
    return redone_transform(img)


_IncompatibleKeys(missing_keys=['idx_queue'], unexpected_keys=['text_encoder.cls.predictions.bias', 'text_encoder.cls.predictions.transform.dense.weight', 'text_encoder.cls.predictions.transform.dense.bias', 'text_encoder.cls.predictions.transform.LayerNorm.weight', 'text_encoder.cls.predictions.transform.LayerNorm.bias', 'text_encoder.cls.predictions.decoder.weight', 'text_encoder.cls.predictions.decoder.bias', 'text_encoder_m.cls.predictions.bias', 'text_encoder_m.cls.predictions.transform.dense.weight', 'text_encoder_m.cls.predictions.transform.dense.bias', 'text_encoder_m.cls.predictions.transform.LayerNorm.weight', 'text_encoder_m.cls.predictions.transform.LayerNorm.bias', 'text_encoder_m.cls.predictions.decoder.weight', 'text_encoder_m.cls.predictions.decoder.bias'])


In [18]:
# ---- 7. Text/image encoding helpers (with tqdm) ---------------------------
@torch.no_grad()
def encode_texts_redone(texts, model, tokenizer, device, batch_size=256, max_length=64):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding candidate pool (ReDonE)'):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding='max_length', truncation=True, max_length=max_length, return_tensors='pt').to(device)
        text_output = model.text_encoder(enc.input_ids, attention_mask=enc.attention_mask, mode='text')
        text_feat = text_output.last_hidden_state[:, 0, :]
        text_embed = F.normalize(model.text_proj(text_feat), dim=-1)
        embs.append(text_embed.cpu())
    return torch.cat(embs, dim=0)

@torch.no_grad()
def encode_images_redone(image_paths, model, device, batch_size=32):
    embs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='Encoding images (ReDonE)'):
        batch_paths = image_paths[i:i+batch_size]
        imgs = torch.stack([load_image_for_redone(p) for p in batch_paths]).to(device)
        image_feat = model.visual_encoder(imgs)
        image_embed = F.normalize(model.vision_proj(image_feat[:, 0, :]), dim=-1)
        embs.append(image_embed.cpu())
    return torch.cat(embs, dim=0)
# ---- 8. Encode candidate pool (text) ---------------------------------------
txt_norm_pool_redone = encode_texts_redone(candidate_pool, redone_model, redone_tokenizer, device).to(device)

# ---- 9. Single-image top-k retrieval ---------------------------------------
def retrieve_topk_redone(img_path, k=10):
    img_emb = encode_images_redone([img_path], redone_model, device, batch_size=1).to(device)
    sims = (img_emb @ txt_norm_pool_redone.t()).squeeze(0)
    topk = torch.topk(sims, k=k)
    sents = [candidate_pool[idx] for idx in topk.indices.tolist()]
    scores = topk.values.tolist()
    return sents, scores

Encoding candidate pool (ReDonE): 100%|██████████| 508/508 [03:56<00:00,  2.15it/s]


# **Test run for all 3 models**

In [20]:
# # ---------------- TEST: retrieval on one image (my model) ----------------
test_id = eval_df["img_id"].iloc[1]
test_row = eval_df[eval_df["img_id"] == test_id].iloc[0]
test_sents, test_scores = retrieve_topk(test_row["Path"], k=10)
print(f"\nimg_id={test_id}  path={test_row['Path']}")
for rank, (s, sc) in enumerate(zip(test_sents, test_scores), 1):
    print(f"{rank:2d}. {sc:.4f}  {s}")

# ── TEST: retrieval on one image (repair) ────────────────────────
test_id = eval_df["img_id"].iloc[0]
test_row = eval_df[eval_df["img_id"] == test_id].iloc[0]
test_sents, test_scores = retrieve_topk_repair(test_row["Path"], k=10)
print(f"\nimg_id={test_id}  path={test_row['Path']}")
for rank, (s, sc) in enumerate(zip(test_sents, test_scores), 1):
    print(f"{rank:2d}. {sc:.4f}  {s}")

# ---- TEST: retrieval on one image(redone) ------------------------------------------
test_id = eval_df["img_id"].iloc[0]
test_row = eval_df[eval_df["img_id"] == test_id].iloc[0]
test_sents_redone, test_scores_redone = retrieve_topk_redone(test_row["Path"], k=10)

print(f"\nimg_id={test_id}  path={test_row['Path']}")
for rank, (s, sc) in enumerate(zip(test_sents_redone, test_scores_redone), 1):
    print(f"{rank:2d}. {sc:.4f}  {s}")



img_id=75  path=/kaggle/input/datasets/ashery/chexpert/train/patient00035/study1/view1_frontal.jpg
 1. 0.7245  There are no kinks or discontinuities along its course.
 2. 0.7198  It most likely represents a nerve stimulation wire placed by anesthesia.
 3. 0.7083  Slight interval retraction of the dual lumen right sided central venous catheter.
 4. 0.6906  No sinister bony lesion.
 5. 0.6827  Bilateral central catheters terminate in the mid SVC
 6. 0.6811  A portable view the chest demonstrates right and left central venous catheters ending in the upper SVC and right atrium, respectively.
 7. 0.6805  No acute cardiopulmonary abnormality Right chest port tip in proximal right atrium
 8. 0.6772  Artifacts from multiple external lines and supporting devices are present.
 9. 0.6749  External ECG leads.
10. 0.6676  Pulmonary opacities described in the CT report are occult on radiography.


Encoding images (RePaiR): 100%|██████████| 1/1 [00:00<00:00, 68.97it/s]



img_id=0  path=/kaggle/input/datasets/ashery/chexpert/train/patient00001/study1/view1_frontal.jpg
 1. 0.3751  Improving atelectasis at the left base, persistent subcutaneous emphysema in the left chest wall.
 2. 0.3721  Left lateral chest wall skin and small postoperative subcutaneous emphysema are new.
 3. 0.3688  Patient is status post median sternotomy multiple valve repair and CABG.
 4. 0.3683  Stable sternal while fragment in the subcutaneous tissues to the left of midline.
 5. 0.3669  Moderate left chest wall subcutaneous emphysema is stable.
 6. 0.3660  Expected postoperative appearance status post CABG with adequate positioning of various lines and catheters.
 7. 0.3653  The patient is status post median sternotomy, CABG, pacemaker placement, and left axillary lymph node dissection.
 8. 0.3652  Stable and normal postoperative widening of the cardiomediastinal silhouette, as expected.
 9. 0.3645  Postoperative increase of heart size to moderate degree, no evidence of pulmonary 

Encoding images (ReDonE): 100%|██████████| 1/1 [00:00<00:00, 30.35it/s]


img_id=0  path=/kaggle/input/datasets/ashery/chexpert/train/patient00001/study1/view1_frontal.jpg
 1. 0.4453  Comparison of portable chest examination suggests some improvement of pulmonary vascular congestion.
 2. 0.4452  Mediastinal widening is likely due to a combination of vascular engorgement and mediastinal fat deposition.
 3. 0.4378  The patient has not returned to his baseline of
 4. 0.4358  No pulmonary edema
 5. 0.4313  Exam is overall unchanged
 6. 0.4301  NO LONGER VISUALIZED.
 7. 0.4290  The cardiac silhouette is
 8. 0.4261  Attention on follow up imaging is recommended.
 9. 0.4258  The image was obtained in lordotic position somewhat limiting evaluation.
10. 0.4249  Widened mediastinum is likely due to a combination of vascular engorgement and mediastinal fat deposition.


# **Full retrieval pipeline**

In [ ]:
from tqdm.auto import tqdm

retrieval_results = {
    img_id: {} for img_id in eval_df["img_id"]
}

for _, r in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Ours"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk(r["Path"], k=10)
    retrieval_results[img_id]["ours"] = {
        "sentences": sents,
        "scores": scores
    }

for _, r in tqdm(eval_df.iterrows(), total=len(eval_df), desc="RePaiR"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk_repair(r["Path"], k=10)
    retrieval_results[img_id]["repair"] = {
        "sentences": sents,
        "scores": scores
    }

for _, r in tqdm(eval_df.iterrows(), total=len(eval_df), desc="ReDonE"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk_redone(r["Path"], k=10)
    retrieval_results[img_id]["redone"] = {
        "sentences": sents,
        "scores": scores
    }

import json

with open("/kaggle/working/retrieval_results.json", "w") as f:
    json.dump(retrieval_results, f)

In [21]:
import json, os, sys, subprocess
with open("/kaggle/input/datasets/shery432/10000ret/retrieval_results.json", "r") as f:
    retrieval_results = json.load(f)

# **Chexbert**

In [22]:
# ---------------- load CheXbert (run once) ----------------
if not os.path.exists('/kaggle/working/CheXbert/src'):
    subprocess.run(['git', 'clone', 'https://github.com/stanfordmlgroup/CheXbert.git',
                     '/kaggle/working/CheXbert'], check=True)

src_path = '/kaggle/working/CheXbert/src'
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

bert_file = os.path.join(src_path, 'models', 'bert_labeler.py')
spec = importlib.util.spec_from_file_location("bert_labeler", bert_file)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
bert_labeler = mod.bert_labeler

CHEXBERT_PATH = '/kaggle/input/datasets/positivecoder/chexbert-model-pth-file/chexbert.pth'
cx_tok = BertTokenizer.from_pretrained('bert-base-uncased')

_orig_from_pretrained = BertModel.from_pretrained
def _fast_from_pretrained(name, *args, **kwargs):
    return BertModel(BertConfig.from_pretrained(name))
BertModel.from_pretrained = _fast_from_pretrained
cx_mdl = bert_labeler()
BertModel.from_pretrained = _orig_from_pretrained

ckpt = torch.load(CHEXBERT_PATH, map_location='cpu')
state = OrderedDict((k[7:] if k.startswith('module.') else k, v) for k, v in ckpt['model_state_dict'].items())
cx_mdl.load_state_dict(state, strict=True)
cx_mdl.eval()
if torch.cuda.is_available():
    cx_mdl = cx_mdl.cuda()

CHEXBERT_CONDITIONS = label_cols  # same 14 names, same order

@torch.no_grad()
def chexbert_label_sentences(sentences, batch_size=32, show_progress=False):
    all_preds = []
    device = next(cx_mdl.parameters()).device
    iterator = range(0, len(sentences), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc='CheXbert labeling', leave=False)
    for i in iterator:
        batch = sentences[i:i+batch_size]
        enc = cx_tok(batch, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        logits = cx_mdl(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        batch_preds = [torch.argmax(l, dim=1).cpu().tolist() for l in logits]
        for j in range(len(batch)):
            all_preds.append({CHEXBERT_CONDITIONS[c]: batch_preds[c][j] for c in range(14)})
    return all_preds

def decisive_labels(pred_dict):
    return {c: ('positive' if v == 1 else 'negative') for c, v in pred_dict.items() if v in (1, 2)}

def positive_labels(pred_dict):
    return {c for c, v in decisive_labels(pred_dict).items() if v == 'positive'}

def negative_labels(pred_dict):
    return {c for c, v in decisive_labels(pred_dict).items() if v == 'negative'}

def union_at_k(cx_preds, k, fn):
    union = set()
    for pred in cx_preds[:k]:
        union |= fn(pred)
    return union

Cloning into '/kaggle/working/CheXbert'...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
from tqdm import tqdm  # NOT tqdm.auto — plain tqdm avoids the widget/ipywidgets rendering issue

chexbert_results = {
    img_id: {} for img_id in retrieval_results
}

for img_id, res in tqdm(retrieval_results.items(), desc="CheXbert Ours", mininterval=2.0):
    chexbert_results[img_id]["ours"] = chexbert_label_sentences(res["ours"]["sentences"])

for img_id, res in tqdm(retrieval_results.items(), desc="CheXbert RePaiR", mininterval=2.0):
    chexbert_results[img_id]["repair"] = chexbert_label_sentences(res["repair"]["sentences"])

for img_id, res in tqdm(retrieval_results.items(), desc="CheXbert ReDonE", mininterval=2.0):
    chexbert_results[img_id]["redone"] = chexbert_label_sentences(res["redone"]["sentences"])

import json
with open("/kaggle/working/chexbert_results.json", "w") as f:
    json.dump(chexbert_results, f)

In [24]:
with open("/kaggle/input/datasets/shery432/10000chex/chexbert_results.json", "r") as f:
    chexbert_results = json.load(f)

# ---- Diagnostic: inspect chexbert_results and gt dicts ----
print("chexbert_results size:", len(chexbert_results))

k1 = next(iter(chexbert_results.keys()))
print("chexbert_results key example:", repr(k1), type(k1))

sample_val = chexbert_results[k1]
print("chexbert_results[img_id] keys:", list(sample_val.keys()))

one_model_preds = sample_val[list(sample_val.keys())[0]]
print("type of preds for one model:", type(one_model_preds))
print("preds sample (first 2 entries):", one_model_preds[:2] if isinstance(one_model_preds, list) else one_model_preds)

print()
print("gt_positive_by_id size:", len(gt_positive_by_id))
k2 = next(iter(gt_positive_by_id.keys()))
print("gt_positive_by_id key example:", repr(k2), type(k2))

print()
overlap = set(chexbert_results.keys()) & set(gt_positive_by_id.keys())
print("overlapping keys:", len(overlap))

chexbert_results size: 10000
chexbert_results key example: '0' <class 'str'>
chexbert_results[img_id] keys: ['ours', 'repair', 'redone']
type of preds for one model: <class 'list'>
preds sample (first 2 entries): [{'No Finding': 1, 'Enlarged Cardiomediastinum': 0, 'Cardiomegaly': 0, 'Lung Opacity': 0, 'Lung Lesion': 0, 'Edema': 0, 'Consolidation': 0, 'Pneumonia': 0, 'Atelectasis': 0, 'Pneumothorax': 0, 'Pleural Effusion': 0, 'Pleural Other': 0, 'Fracture': 0, 'Support Devices': 0}, {'No Finding': 0, 'Enlarged Cardiomediastinum': 0, 'Cardiomegaly': 0, 'Lung Opacity': 0, 'Lung Lesion': 0, 'Edema': 0, 'Consolidation': 0, 'Pneumonia': 0, 'Atelectasis': 0, 'Pneumothorax': 0, 'Pleural Effusion': 0, 'Pleural Other': 0, 'Fracture': 0, 'Support Devices': 1}]

gt_positive_by_id size: 10000
gt_positive_by_id key example: 0 <class 'int'>

overlapping keys: 0


In [25]:
# ============================================================
# FIX + Metrics: Recall / Precision / F1 @ K, overall + per-label
# ============================================================
import json

# ---- 0. Fix key types (str -> int) ----
chexbert_results = {int(k): v for k, v in chexbert_results.items()}

# sanity check overlap now
overlap = set(chexbert_results.keys()) & set(gt_positive_by_id.keys())
print(f"overlapping keys after fix: {len(overlap)} / {len(chexbert_results)}")

# ---- 1. Label helpers (data is already binary: 1=positive, 0=not) ----
def positive_labels(pred_dict):
    return {c for c, v in pred_dict.items() if v == 1}

def union_at_k(cx_preds, k, fn):
    union = set()
    for pred in cx_preds[:k]:
        union |= fn(pred)
    return union

# ---- 2. Core metric computation ----
MODEL_DISPLAY_NAMES = {
    "ours":   "My Model Results baseline using JOImTer",
    "repair": "CXR-RePaiR Results",
    "redone": "CXR-ReDonE Results",
}
K_VALUES = (1, 5, 10)

def compute_metrics(chexbert_results, gt_positive_by_id, gt_usable_by_id,
                     label_cols, model_key, k_values=K_VALUES):
    n = 0
    overall = {k: {'tp': 0, 'fp': 0, 'fn': 0} for k in k_values}
    per_label = {k: {lbl: {'tp': 0, 'fp': 0, 'fn': 0} for lbl in label_cols} for k in k_values}

    for img_id, preds_by_model in chexbert_results.items():
        if img_id not in gt_positive_by_id:
            continue
        gt_usable = gt_usable_by_id[img_id]
        if not gt_usable:
            continue
        cx_preds = preds_by_model[model_key]
        gt_pos = gt_positive_by_id[img_id]
        n += 1

        for k in k_values:
            pred_pos_k = union_at_k(cx_preds, k, positive_labels)
            for lbl in label_cols:
                if lbl not in gt_usable:
                    continue
                is_pred_pos = lbl in pred_pos_k
                is_gt_pos = lbl in gt_pos
                if is_pred_pos and is_gt_pos:
                    overall[k]['tp'] += 1
                    per_label[k][lbl]['tp'] += 1
                elif is_pred_pos and not is_gt_pos:
                    overall[k]['fp'] += 1
                    per_label[k][lbl]['fp'] += 1
                elif not is_pred_pos and is_gt_pos:
                    overall[k]['fn'] += 1
                    per_label[k][lbl]['fn'] += 1

    return overall, per_label, n

def _prf(tp, fp, fn):
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return recall, precision, f1

# ---- run for all three models ----
results = {}
for model_key in ["ours", "repair", "redone"]:
    overall, per_label, n = compute_metrics(
        chexbert_results, gt_positive_by_id, gt_usable_by_id, label_cols, model_key
    )
    results[model_key] = {"overall": overall, "per_label": per_label, "n": n}

# ============================================================
# TABLE 1: Overall Recall / Precision / F1 @K
# ============================================================
print("=== TABLE 1: Overall Recall / Precision / F1 @K=1,5,10 ===\n")
for model_key in ["ours", "repair", "redone"]:
    r = results[model_key]
    print(f"{MODEL_DISPLAY_NAMES[model_key]}:")
    for k in K_VALUES:
        tp, fp, fn = r["overall"][k]['tp'], r["overall"][k]['fp'], r["overall"][k]['fn']
        recall, precision, f1 = _prf(tp, fp, fn)
        print(f"K={k}: Recall={recall:.4f} Precision={precision:.4f} F1={f1:.4f} (n={r['n']})")
    print()

# ============================================================
# TABLE 2: Per-label TP/FP/FN/Recall/Precision/F1 @K=5 (per model)
# ============================================================
print("=== TABLE 2: Per-label TP/FP/FN/Recall/Precision/F1 @K=5 (per model) ===\n")
K_FOR_TABLE2 = 5
SHORT_NAMES = {"ours": "Yours", "repair": "CXR-RePaiR", "redone": "CXR-ReDonE"}

for model_key in ["ours", "repair", "redone"]:
    r = results[model_key]
    print(f"-- {SHORT_NAMES[model_key]} --")
    print(f"{'Label':<28}{'TP':>5}{'FP':>5}{'FN':>5}{'Recall':>9}{'Prec':>8}{'F1':>8}")
    for lbl in label_cols:
        tp = r["per_label"][K_FOR_TABLE2][lbl]['tp']
        fp = r["per_label"][K_FOR_TABLE2][lbl]['fp']
        fn = r["per_label"][K_FOR_TABLE2][lbl]['fn']
        recall, precision, f1 = _prf(tp, fp, fn)
        print(f"{lbl:<28}{tp:>5}{fp:>5}{fn:>5}{recall:>9.3f}{precision:>8.3f}{f1:>8.3f}")
    print()

# ============================================================
# TABLE 3: Labels ranked by Recall (lowest = hardest) @K=5
# ============================================================
print("=== TABLE 3: Labels ranked by Recall (lowest = hardest) @K=5 ===\n")
K_FOR_TABLE3 = 5
EXCLUDE_LABELS = {"No Finding"}  # matches your target output; remove to include it

for model_key in ["ours", "repair", "redone"]:
    r = results[model_key]
    print(f"-- {SHORT_NAMES[model_key]} (hardest first) --")

    label_recalls = []
    for lbl in label_cols:
        if lbl in EXCLUDE_LABELS:
            continue
        tp = r["per_label"][K_FOR_TABLE3][lbl]['tp']
        fn = r["per_label"][K_FOR_TABLE3][lbl]['fn']
        support = tp + fn
        if support == 0:
            continue  # no positive cases for this label — recall undefined, skip
        recall = tp / support
        label_recalls.append((lbl, recall))

    label_recalls.sort(key=lambda x: x[1])  # ascending: lowest recall = hardest first

    for rank, (lbl, recall) in enumerate(label_recalls, 1):
        print(f"  {rank:2d}. {lbl}: Recall={recall:.3f}")
    print()

overlapping keys after fix: 10000 / 10000
=== TABLE 1: Overall Recall / Precision / F1 @K=1,5,10 ===

My Model Results baseline using JOImTer:
K=1: Recall=0.1485 Precision=0.2938 F1=0.1973 (n=10000)
K=5: Recall=0.2594 Precision=0.2333 F1=0.2456 (n=10000)
K=10: Recall=0.3055 Precision=0.2099 F1=0.2488 (n=10000)

CXR-RePaiR Results:
K=1: Recall=0.0890 Precision=0.1496 F1=0.1116 (n=10000)
K=5: Recall=0.2191 Precision=0.1660 F1=0.1889 (n=10000)
K=10: Recall=0.2908 Precision=0.1694 F1=0.2141 (n=10000)

CXR-ReDonE Results:
K=1: Recall=0.1364 Precision=0.2894 F1=0.1854 (n=10000)
K=5: Recall=0.2429 Precision=0.2382 F1=0.2405 (n=10000)
K=10: Recall=0.2832 Precision=0.2160 F1=0.2450 (n=10000)

=== TABLE 2: Per-label TP/FP/FN/Recall/Precision/F1 @K=5 (per model) ===

-- Yours --
Label                          TP   FP   FN   Recall    Prec      F1
No Finding                    116 1454  779    0.130   0.074   0.094
Enlarged Cardiomediastinum     62  901  450    0.121   0.064   0.084
Cardiomegaly  

# **Pool Composition**

In [26]:
# ============================================================
# Pool composition test: label the full candidate pool with
# CheXbert and report per-label positive sentence counts
# ============================================================
from tqdm import tqdm

@torch.no_grad()
def chexbert_label_sentences_tqdm(sentences, batch_size=64):
    all_preds = []
    device = next(cx_mdl.parameters()).device
    for i in tqdm(range(0, len(sentences), batch_size), desc='Labeling candidate pool'):
        batch = sentences[i:i+batch_size]
        enc = cx_tok(batch, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        logits = cx_mdl(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        batch_preds = [torch.argmax(l, dim=1).cpu().tolist() for l in logits]
        for j in range(len(batch)):
            all_preds.append({CHEXBERT_CONDITIONS[c]: batch_preds[c][j] for c in range(14)})
    return all_preds

pool_preds = chexbert_label_sentences_tqdm(candidate_pool, batch_size=64)
# pool_preds: list of dicts, one per sentence in candidate_pool, e.g.
# {'No Finding': 1, 'Cardiomegaly': 0, ...}

# ---- tally positive counts per label ----
label_counts = {lbl: 0 for lbl in label_cols}
for pred in pool_preds:
    for lbl in label_cols:
        if pred[lbl] == 1:
            label_counts[lbl] += 1

total = len(candidate_pool)

# ---- print sorted by count, descending ----
print(f"=== Pool composition: {total} sentences ===\n")
print(f"{'Label':<28}{'Count':>8}{'% of pool':>12}")
for lbl, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * count / total
    print(f"{lbl:<28}{count:>8}{pct:>11.2f}%")

Labeling candidate pool: 100%|██████████| 2030/2030 [15:13<00:00,  2.22it/s]

=== Pool composition: 129906 sentences ===

Label                          Count   % of pool
Support Devices                50701      39.03%
Fracture                       24209      18.64%
Cardiomegaly                   22506      17.32%
Pneumothorax                   13948      10.74%
Pneumonia                      12784       9.84%
Lung Lesion                     8805       6.78%
Enlarged Cardiomediastinum      6199       4.77%
No Finding                      3227       2.48%
Consolidation                   2580       1.99%
Edema                           2440       1.88%
Atelectasis                     2371       1.83%
Lung Opacity                    1631       1.26%
Pleural Other                   1585       1.22%
Pleural Effusion                 530       0.41%


# **Build Label-Balanced Candidate Pool**

In [30]:
import numpy as np

SEED = 42
CAP_PER_LABEL = 450  # bounded by smallest class: Pleural Effusion has 530 total
rng = np.random.default_rng(SEED)

# group candidate_pool indices by label (multi-label overlap allowed)
label_to_indices = {lbl: [] for lbl in label_cols}
for i, pred in enumerate(pool_preds):
    for lbl in label_cols:
        if pred[lbl] == 1:
            label_to_indices[lbl].append(i)

print(f"{'Label':<28}{'Available':>10}{'Sampled':>10}")
balanced_indices = set()
for lbl in label_cols:
    avail = label_to_indices[lbl]
    n_take = min(CAP_PER_LABEL, len(avail))
    if n_take < len(avail):
        chosen = rng.choice(avail, size=n_take, replace=False).tolist()
    else:
        chosen = avail  # take everything if fewer than cap (flag these as capped-low)
    balanced_indices.update(chosen)
    flag = "  <-- capped by availability" if n_take < CAP_PER_LABEL else ""
    print(f"{lbl:<28}{len(avail):>10}{n_take:>10}{flag}")

balanced_indices = sorted(balanced_indices)
balanced_pool = [candidate_pool[i] for i in balanced_indices]
print(f"\nBalanced pool total size (deduped, multi-label overlap collapsed): {len(balanced_pool)}")

Label                        Available   Sampled
No Finding                        3227       450
Enlarged Cardiomediastinum        6199       450
Cardiomegaly                     22506       450
Lung Opacity                      1631       450
Lung Lesion                       8805       450
Edema                             2440       450
Consolidation                     2580       450
Pneumonia                        12784       450
Atelectasis                       2371       450
Pneumothorax                     13948       450
Pleural Effusion                   530       450
Pleural Other                     1585       450
Fracture                         24209       450
Support Devices                  50701       450

Balanced pool total size (deduped, multi-label overlap collapsed): 6216


In [33]:
balanced_preds = chexbert_label_sentences_tqdm(balanced_pool, batch_size=64)

balanced_label_counts = {lbl: 0 for lbl in label_cols}
for pred in balanced_preds:
    for lbl in label_cols:
        if pred[lbl] == 1:
            balanced_label_counts[lbl] += 1

total_b = len(balanced_pool)
print(f"=== Balanced pool composition: {total_b} sentences ===\n")
print(f"{'Label':<28}{'Count':>8}{'% of pool':>12}")
for lbl, count in sorted(balanced_label_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * count / total_b
    print(f"{lbl:<28}{count:>8}{pct:>11.2f}%")

Labeling candidate pool: 100%|██████████| 98/98 [00:44<00:00,  2.22it/s]

=== Balanced pool composition: 6216 sentences ===

Label                          Count   % of pool
Cardiomegaly                    1345      21.64%
Pneumothorax                     989      15.91%
Support Devices                  859      13.82%
Pneumonia                        819      13.18%
Fracture                         781      12.56%
Lung Lesion                      731      11.76%
Enlarged Cardiomediastinum       575       9.25%
Consolidation                    573       9.22%
Edema                            568       9.14%
No Finding                       500       8.04%
Lung Opacity                     487       7.83%
Atelectasis                      476       7.66%
Pleural Other                    457       7.35%
Pleural Effusion                 453       7.29%


In [31]:
candidate_pool_full = candidate_pool   # keep the original 129,906-sentence pool safe
candidate_pool = balanced_pool         # retrieval functions now index into this

print(f"Original pool: {len(candidate_pool_full)} sentences")
print(f"Balanced pool: {len(candidate_pool)} sentences (now active for retrieval)")

BATCH = 256
all_sent_feats = []
with torch.no_grad():
    for i in tqdm(range(0, len(candidate_pool), BATCH), desc='Encoding balanced pool (Ours)'):
        batch_sents = candidate_pool[i:i+BATCH]
        enc = tokenizer(batch_sents, padding='max_length', truncation=True, max_length=64, return_tensors='pt').to(DEVICE)
        _, sent_feats = text_encoder(enc['input_ids'], enc['attention_mask'], task='itm')
        all_sent_feats.append(sent_feats.cpu())
all_sent_feats = torch.cat(all_sent_feats, dim=0)
txt_norm_pool = torch.nn.functional.normalize(all_sent_feats, dim=1).to(DEVICE)

txt_norm_pool_repair = encode_texts_repair(candidate_pool, repair_model, device).to(device)

txt_norm_pool_redone = encode_texts_redone(candidate_pool, redone_model, redone_tokenizer, device).to(device)

print("All 3 pool embeddings re-encoded against the balanced pool.")

Original pool: 129906 sentences
Balanced pool: 6216 sentences (now active for retrieval)


Encoding candidate pool (ReDonE): 100%|██████████| 25/25 [00:11<00:00,  2.18it/s]

All 3 pool embeddings re-encoded against the balanced pool.


In [37]:
from tqdm.auto import tqdm as tqdm_auto

retrieval_results_balanced = {
    img_id: {} for img_id in eval_df["img_id"]
}

for _, r in tqdm_auto(eval_df.iterrows(), total=len(eval_df), desc="Ours (balanced)"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk(r["Path"], k=10)
    retrieval_results_balanced[img_id]["ours"] = {"sentences": sents, "scores": scores}

for _, r in tqdm_auto(eval_df.iterrows(), total=len(eval_df), desc="RePaiR (balanced)"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk_repair(r["Path"], k=10)
    retrieval_results_balanced[img_id]["repair"] = {"sentences": sents, "scores": scores}

for _, r in tqdm_auto(eval_df.iterrows(), total=len(eval_df), desc="ReDonE (balanced)"):
    img_id = r["img_id"]
    sents, scores = retrieve_topk_redone(r["Path"], k=10)
    retrieval_results_balanced[img_id]["redone"] = {"sentences": sents, "scores": scores}

import json
with open("/kaggle/working/retrieval_results_balanced.json", "w") as f:
    json.dump(retrieval_results_balanced, f)

Ours (balanced):   0%|          | 0/10000 [00:00<?, ?it/s]

RePaiR (balanced):   0%|          | 0/10000 [00:00<?, ?it/s]

ReDonE (balanced):   0%|          | 0/10000 [00:00<?, ?it/s]

In [38]:
sentence_to_label = {s: balanced_preds[i] for i, s in enumerate(balanced_pool)}

def label_via_lookup(sentences):
    out = []
    missing = []
    for s in sentences:
        if s in sentence_to_label:
            out.append(sentence_to_label[s])
        else:
            missing.append(s)
            out.append(None)  # placeholder, filled below
    if missing:
        fresh = chexbert_label_sentences(missing)
        it = iter(fresh)
        out = [sentence_to_label.get(s) if s in sentence_to_label else next(it) for s in sentences]
    return out

chexbert_results_balanced = {
    img_id: {} for img_id in retrieval_results_balanced
}

for img_id, res in tqdm(retrieval_results_balanced.items(), desc="Labeling Ours (lookup)", mininterval=2.0):
    chexbert_results_balanced[img_id]["ours"] = label_via_lookup(res["ours"]["sentences"])

for img_id, res in tqdm(retrieval_results_balanced.items(), desc="Labeling RePaiR (lookup)", mininterval=2.0):
    chexbert_results_balanced[img_id]["repair"] = label_via_lookup(res["repair"]["sentences"])

for img_id, res in tqdm(retrieval_results_balanced.items(), desc="Labeling ReDonE (lookup)", mininterval=2.0):
    chexbert_results_balanced[img_id]["redone"] = label_via_lookup(res["redone"]["sentences"])

with open("/kaggle/working/chexbert_results_balanced.json", "w") as f:
    json.dump(chexbert_results_balanced, f)

Labeling ReDonE (lookup): 100%|██████████| 10000/10000 [00:00<00:00, 351546.73it/s]


In [39]:
results_balanced = {}
for model_key in ["ours", "repair", "redone"]:
    overall, per_label, n = compute_metrics(
        chexbert_results_balanced, gt_positive_by_id, gt_usable_by_id, label_cols, model_key
    )
    results_balanced[model_key] = {"overall": overall, "per_label": per_label, "n": n}

print("=== BALANCED POOL -- TABLE 1: Overall Recall / Precision / F1 @K=1,5,10 ===\n")
for model_key in ["ours", "repair", "redone"]:
    r = results_balanced[model_key]
    print(f"{MODEL_DISPLAY_NAMES[model_key]}:")
    for k in K_VALUES:
        tp, fp, fn = r["overall"][k]['tp'], r["overall"][k]['fp'], r["overall"][k]['fn']
        recall, precision, f1 = _prf(tp, fp, fn)
        print(f"K={k}: Recall={recall:.4f} Precision={precision:.4f} F1={f1:.4f} (n={r['n']})")
    print()

print("=== BALANCED POOL -- TABLE 2: Per-label TP/FP/FN/Recall/Precision/F1 @K=5 ===\n")
K_FOR_TABLE2 = 5
for model_key in ["ours", "repair", "redone"]:
    r = results_balanced[model_key]
    print(f"-- {SHORT_NAMES[model_key]} --")
    print(f"{'Label':<28}{'TP':>5}{'FP':>5}{'FN':>5}{'Recall':>9}{'Prec':>8}{'F1':>8}")
    for lbl in label_cols:
        tp = r["per_label"][K_FOR_TABLE2][lbl]['tp']
        fp = r["per_label"][K_FOR_TABLE2][lbl]['fp']
        fn = r["per_label"][K_FOR_TABLE2][lbl]['fn']
        recall, precision, f1 = _prf(tp, fp, fn)
        print(f"{lbl:<28}{tp:>5}{fp:>5}{fn:>5}{recall:>9.3f}{precision:>8.3f}{f1:>8.3f}")
    print()

=== BALANCED POOL -- TABLE 1: Overall Recall / Precision / F1 @K=1,5,10 ===

My Model Results baseline using JOImTer:
K=1: Recall=0.1215 Precision=0.2018 F1=0.1517 (n=10000)
K=5: Recall=0.3030 Precision=0.1900 F1=0.2335 (n=10000)
K=10: Recall=0.4035 Precision=0.1843 F1=0.2530 (n=10000)

CXR-RePaiR Results:
K=1: Recall=0.1049 Precision=0.1388 F1=0.1195 (n=10000)
K=5: Recall=0.2543 Precision=0.1501 F1=0.1888 (n=10000)
K=10: Recall=0.3464 Precision=0.1574 F1=0.2164 (n=10000)

CXR-ReDonE Results:
K=1: Recall=0.1233 Precision=0.2084 F1=0.1549 (n=10000)
K=5: Recall=0.2817 Precision=0.1897 F1=0.2267 (n=10000)
K=10: Recall=0.3685 Precision=0.1797 F1=0.2416 (n=10000)

=== BALANCED POOL -- TABLE 2: Per-label TP/FP/FN/Recall/Precision/F1 @K=5 ===

-- Yours --
Label                          TP   FP   FN   Recall    Prec      F1
No Finding                    266 3875  629    0.297   0.064   0.106
Enlarged Cardiomediastinum    145 2455  367    0.283   0.056   0.093
Cardiomegaly                  430 